In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/eur_weekly.csv")

df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,04/05/2026,1.1720,1.1518,1.1740,1.1505,NaN,1.76%
1,03/29/2026,1.1517,1.1518,1.1628,1.1443,NaN,0.06%
2,03/22/2026,1.1510,1.1542,1.1640,1.1484,NaN,-0.53%
3,03/15/2026,1.1571,1.1414,1.1616,1.1412,NaN,1.36%
4,03/08/2026,1.1416,1.1569,1.1669,1.1411,NaN,-1.74%


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      1371 non-null   str    
 1   Price     1371 non-null   float64
 2   Open      1371 non-null   float64
 3   High      1371 non-null   float64
 4   Low       1371 non-null   float64
 5   Vol.      0 non-null      float64
 6   Change %  1371 non-null   str    
dtypes: float64(5), str(2)
memory usage: 75.1 KB


# Descriptive Statistics

This section provides an initial statistical summary of the numerical variables in the dataset. Examining descriptive statistics helps identify the range, central tendency, and variability of the data before preprocessing and model development.

In [3]:
df.describe()

,Price,Open,High,Low,Vol.
count,1371.000000,1371.000000,1371.000000,1371.000000,0.0
mean,1.186985,1.186793,1.198623,1.175265,NaN
std,0.153382,0.153544,0.154893,0.151804,NaN
min,0.838600,0.839200,0.844300,0.822700,NaN
25%,1.088900,1.089200,1.099650,1.080550,NaN
50%,1.175200,1.175000,1.183800,1.166300,NaN
75%,1.300800,1.300300,1.314600,1.288050,NaN
max,1.594200,1.596400,1.604000,1.578400,NaN


## Interpretation of Descriptive Statistics

The descriptive statistics provide an initial overview of the numerical variables in the EUR/USD weekly dataset.

Key observations:

- The dataset contains **1,371 weekly observations**.
- The average closing exchange rate (`Price`) is approximately **1.1870**.
- Exchange rates range from **0.8386** to **1.5942**, indicating substantial variation over the study period.
- The `Open`, `High`, and `Low` variables exhibit similar distributions, which is expected for OHLC financial data.
- The `Vol.` column contains no observations and is therefore likely to be removed during the preprocessing stage.

## Data Quality Assessment

Before preprocessing the dataset, it is important to evaluate its quality. This includes checking for missing values and duplicate records that may affect the reliability of the analysis and the performance of forecasting models.

In [4]:
df.isnull().sum()

Date           0
Price          0
Open           0
High           0
Low            0
Vol.        1371
Change %       0
dtype: int64

## Interpretation of Missing Values

The missing value assessment indicates that the dataset is generally complete, with the exception of the `Vol.` column.

Key observations:

- The `Date`, `Price`, `Open`, `High`, `Low`, and `Change %` columns contain no missing values.
- The `Vol.` column contains **1,371 missing values**, meaning that every observation is missing.
- Since the `Vol.` column provides no usable information, it will be removed during the preprocessing stage.

In [5]:
df.duplicated().sum()

np.int64(0)

## Interpretation of Duplicate Records

The duplicate record assessment indicates that the dataset contains **no duplicate observations**.

This confirms that each weekly record is unique and no duplicate rows need to be removed before further preprocessing or analysis.

## Converting the Date Column

The `Date` column is currently stored as text (`string`). Before performing any time-series analysis, it must be converted to a datetime format so that Python can correctly interpret dates, sort observations chronologically, and support time-based operations.

In [6]:
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      1371 non-null   datetime64[us]
 1   Price     1371 non-null   float64       
 2   Open      1371 non-null   float64       
 3   High      1371 non-null   float64       
 4   Low       1371 non-null   float64       
 5   Vol.      0 non-null      float64       
 6   Change %  1371 non-null   str           
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 75.1 KB


## Interpretation of Date Conversion

The `Date` column was successfully converted from a text data type to the `datetime` format.

This conversion is essential for time-series analysis because it enables chronological sorting, time-based indexing, filtering by date, and the application of forecasting techniques that rely on temporal ordering.

In [7]:
df[["Date"]].head()

,Date
0,2026-04-05
1,2026-03-29
2,2026-03-22
3,2026-03-15
4,2026-03-08


In [8]:
df[["Date"]].tail()

,Date
1366,2000-01-30
1367,2000-01-23
1368,2000-01-16
1369,2000-01-09
1370,2000-01-02


## Chronological Order of the Dataset

The dataset is currently arranged in reverse chronological order, with the most recent observations appearing first.

For time-series forecasting, the data will be reordered into chronological order (oldest to newest). This ensures that all models are trained using historical observations to predict future values, preventing methodological issues related to temporal ordering.

In [9]:
df = df.sort_values("Date").reset_index(drop=True)

df[["Date"]].head()

,Date
0,2000-01-02
1,2000-01-09
2,2000-01-16
3,2000-01-23
4,2000-01-30


In [10]:
df.to_csv("../data/processed/eur_weekly_clean.csv", index=False)

## Removing the Empty Volume Column

The `Vol.` column contains only missing values and therefore provides no useful information for analysis or forecasting. It is removed from the dataset.

In [11]:
df = df.drop(columns=["Vol."])

df.head()

,Date,Price,Open,High,Low,Change %
0,2000-01-02,1.0292,1.0052,1.0417,1.0050,2.15%
1,2000-01-09,1.0128,1.0288,1.0370,1.0111,-1.59%
2,2000-01-16,1.0091,1.0128,1.0193,1.0049,-0.37%
3,2000-01-23,0.9750,1.0026,1.0101,0.9738,-3.38%
4,2000-01-30,0.9828,0.9779,0.9949,0.9662,0.80%


## Converting the Percentage Change Column

The `Change %` column is currently stored as text because each value contains a percentage symbol (`%`). To enable numerical analysis, the percentage symbol is removed and the values are converted to a numeric data type.

In [12]:
df["Change %"] = (
    df["Change %"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      1371 non-null   datetime64[us]
 1   Price     1371 non-null   float64       
 2   Open      1371 non-null   float64       
 3   High      1371 non-null   float64       
 4   Low       1371 non-null   float64       
 5   Change %  1371 non-null   float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 64.4 KB


In [13]:
df.to_csv("../data/processed/eur_weekly_clean.csv", index=False)

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      1371 non-null   datetime64[us]
 1   Price     1371 non-null   float64       
 2   Open      1371 non-null   float64       
 3   High      1371 non-null   float64       
 4   Low       1371 non-null   float64       
 5   Change %  1371 non-null   float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 64.4 KB


In [15]:
check = pd.read_csv("../data/processed/eur_weekly_clean.csv")
check.info()

<class 'pandas.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      1371 non-null   str    
 1   Price     1371 non-null   float64
 2   Open      1371 non-null   float64
 3   High      1371 non-null   float64
 4   Low       1371 non-null   float64
 5   Change %  1371 non-null   float64
dtypes: float64(5), str(1)
memory usage: 64.4 KB


In [16]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
D:\project\sharemark\mainthesis\notebooks


In [17]:
import os

print(os.path.abspath("../data/processed/eur_weekly_clean.csv"))

D:\project\sharemark\mainthesis\data\processed\eur_weekly_clean.csv


In [18]:
os.path.exists("../data/processed/eur_weekly_clean.csv")

True

In [19]:
df.to_csv("../data/processed/eur_weekly_clean.csv", index=False)

In [20]:
check = pd.read_csv("../data/processed/eur_weekly_clean.csv")
check.info()

<class 'pandas.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      1371 non-null   str    
 1   Price     1371 non-null   float64
 2   Open      1371 non-null   float64
 3   High      1371 non-null   float64
 4   Low       1371 non-null   float64
 5   Change %  1371 non-null   float64
dtypes: float64(5), str(1)
memory usage: 64.4 KB
